# Polars Part 3: Electricity Mix and Energy Use

This notebook rebuilds the features needed for the energy sections, then focuses on electricity generation shares and energy-use relationships.

Links:
- [Polars notebook overview](README.md)
- [Shared helpers](../functions.py)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / 'data' / 'co2_data.csv').exists() and (candidate / 'notebooks' / 'functions.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
DATA_DIR = REPO_ROOT / 'data'

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
import umap.umap_ as umap

import timeit
import requests

from functions import (
    apply_correlation_to_df,
    normalize_column,
    z_score_column,
    safe_divide,
    classify_income_group,
    get_top_bottom_n,
    compute_energy_mix_shares,
    plot_dual_axis_timeseries,
    run_kmeans_elbow,
    fetch_wikipedia_gni_table,
    fetch_world_bank_xml_records,
)

# clean plotting defaults
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
})
%matplotlib inline


## Setup

Import libraries and configure plot styling. We use `seaborn` for statistical graphics with a clean whitegrid theme, and `autoreload` to hot-reload helper functions from [functions.py](../functions.py) during development.

# Carbon Emissions and Economic Development: A Visual Analysis

The central question driving this analysis is whether economic growth necessarily comes at the cost of rising carbon emissions - or whether countries can **decouple** the two. This is one of the most consequential questions in climate policy: if decoupling is possible, it suggests that prosperity and sustainability are not mutually exclusive.

To investigate this, we combine two authoritative datasets:

1. **CO2 Emissions** (Our World in Data, 1750–2024) - historical emissions by country, measured in millions of tonnes
2. **GDP** (World Bank, 1960–2024) - economic output in current USD
3. **Electricity** (Our World in Data, 2000-2024) - Share of electricity production methods of the countries

By normalizing both metrics to per-capita values and computing Pearson correlations over time, we can classify countries along a spectrum: from those where growth and emissions move in lockstep (strong coupling) to those where GDP continues to rise while emissions fall (decoupling). We then ask whether this pattern is systematic - do high-income countries decouple more than low-income ones?

## CO2 Emissions Data

We load the Our World in Data CO2 dataset, which contains 79 columns spanning energy mix, land use, and emissions breakdowns. For this analysis we reduce it to five key variables: `country`, `year`, `iso_code`, `population`, and `co2` (total production-based CO2 emissions in millions of tonnes).

Two important filtering steps:
- **Drop rows without `iso_code`**: The dataset includes aggregate entities like "Africa", "OECD", and "World" that lack ISO country codes. Removing these ensures we work exclusively with individual nation-states.
- **Filter to post-1960**: GDP data from the World Bank only begins in 1960, so we align the time range to enable a clean merge later.

In [ ]:
co2_df = pl.read_csv(DATA_DIR / 'co2_data.csv')

# choose what columns to keep (can be changed, but this is the most important)
selected_columns = ['country', 'year', 'iso_code', 'population', 'co2']

# drop the columns that are not in the selected_columns list
co2_df = co2_df.select(selected_columns)

# remove entries with no iso code(Continents and other groups)
co2_df = (
    co2_df
    .with_columns(pl.col("co2")
                    .str.strip_chars()
                    .replace("", None)
                    .cast(pl.Float64))
    
    .filter(pl.col("iso_code") != "")
    .filter(pl.col("year") > 1960))


co2_df


## GDP Data

The World Bank GDP dataset arrives in **wide format** - one column per year (1960, 1961, ..., 2024). This is convenient for spreadsheet viewing but incompatible with tidy-data principles needed for plotting and merging. We use `pl.unipivot()` to reshape it into long format with one row per country-year observation.

Key steps:
- **Melt** year columns into `year` (int) and `gdp` (numeric) columns
- **Rename** `Country Code` to `iso_code` to create a shared merge key with the CO2 dataset
- **Drop missing GDP values** - not all countries have GDP records for every year, particularly in earlier decades or for newly independent states

In [ ]:
# load GDP data
gdp_df = pl.read_csv(DATA_DIR / 'gdp_data.csv')

# reshape from wide to long format
# melt the year columns into rows
year_columns = [col for col in gdp_df.columns if col.isdigit()]
gdp_df = gdp_df.unpivot(
    index=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    on=year_columns,
    variable_name='year',
    value_name='gdp')

# rename Country Code to iso_code for merging
gdp_df = gdp_df.rename({'Country Code': 'iso_code'})

# keep only the columns we need
gdp_df = gdp_df[['iso_code', 'year', 'gdp']]

gdp_df = (gdp_df
          .with_columns(pl.col("year").str.to_integer())
          .with_columns(pl.col("gdp").str.strip_chars().replace("", None).cast(pl.Float64))
          .drop_nulls(subset=["gdp"]))

gdp_df

## Merging the Datasets

We perform a **left join** of GDP onto the CO2 dataframe using `iso_code` and `year` as composite keys. A left join preserves every CO2 record and attaches GDP where available - countries or years without World Bank GDP data simply receive NaN. This is preferable to an inner join because it avoids silently discarding emission records that are still valuable for other analyses.

### Missing Data Heatmap

Before proceeding, we visualize data completeness. The heatmap below shows each variable as a column and each record as a row, with bright cells indicating missing values. This diagnostic is important because it reveals whether missingness is **random** or **systematic** - for instance, GDP data may be consistently absent for certain countries or time periods, which would bias any analysis that silently drops incomplete rows.

In [ ]:
# merge the datasets on iso_code and year
base_df = co2_df.join(
    gdp_df,
    on=['iso_code', 'year'],
    how='left',)

# missing data heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(base_df.select(pl.all().is_null()), yticklabels=False, cbar=False, cmap='magma_r', ax=ax)
ax.set_title("Missing Data Overview After Merge")

plt.tight_layout()
plt.show()

In [ ]:
# return null values per column
base_df.select(pl.all().null_count())

### Data Quality Summary

The GDP column shows the most missingness - this is expected since many countries (especially newly independent or conflict-affected states) lack World Bank GDP records in earlier decades. Importantly, the missingness is **systematic** rather than random: it concentrates in specific countries and time periods. This means any analysis involving GDP will implicitly exclude these observations, so conclusions are most robust for the subset of countries with consistent economic data.

## Per-Capita Metrics

Absolute CO2 and GDP figures are dominated by population size - China and India will always top the charts simply because they have the most people, not necessarily because their economies or industries are more carbon-intensive on a per-person basis. Dividing by population yields **per-capita** values that enable fairer cross-country comparisons: how much does the average citizen emit, and how wealthy is the average citizen?

- **CO2 per capita** is expressed in **tonnes per person** (the raw CO2 column is in millions of tonnes, so we multiply by 10⁶ before dividing by population).
- **GDP per capita** is in **current USD per person**.

We use `np.divide` with a `where` guard to handle zero or missing population entries without raising division errors.

In [ ]:
# CO2 is in millions of tonnes; multiply by 1e6 to get tonnes, then divide by population
base_df = base_df.with_columns(
    (pl.col("co2") / pl.col("population") * 1e6)
    .replace([float("inf"), -float("inf")], None)
    .alias("co2_per_capita"))

base_df = base_df.with_columns(
    (pl.col("gdp") / pl.col("population"))
    .replace([float("inf"), -float("inf")], None)
    .alias("gdp_per_capita"))

# Log-transform per-capita metrics for better visualization
base_df = base_df.with_columns(np.log1p(pl.col('gdp_per_capita'))
                               .alias("log_gdp_pc"))
base_df = base_df.with_columns(np.log1p(pl.col('co2_per_capita'))
                              .alias("log_co2_pc"))                     

# compare raw and log-transformed data as histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(base_df.select("gdp_per_capita"), bins=50, edgecolor='white')
axes[0].set_title('GDP per Capita – Raw Data')
axes[1].hist(base_df.select("log_gdp_pc"), bins=50, edgecolor='white', color='seagreen')
axes[1].set_title('GDP per Capita – Log-Transformed (approx. normal)')

plt.tight_layout()
plt.show()


## Income Group Classification

To move beyond individual country case studies, we classify each observation by **income group** using the World Bank's Gross National Income (GNI) thresholds:

| Group | GNI per capita (current USD) |
|---|---|
| Low | $\leq$ 1,145 |
| Lower-Middle | 1,146 – 4,515 |
| Upper-Middle | 4,516 – 14,005 |
| High | $>$ 14,005 |

Source: [World Bank Country Classification (FY2025)](https://datahelpdesk.worldbank.org/knowledgebase/articles/906519-world-bank-country-and-lending-groups)

This allows us to ask a structural question: do emission trajectories differ systematically between rich and poor countries? We scrape current GNI per capita figures from Wikipedia and merge them onto our dataset.

In [ ]:
# fetch data from wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GNI_(nominal)_per_capita'

# pd.read_html would have been possible too
# but BeautifulSoup gives us more control over the wikipedia table structure
gni_df = fetch_wikipedia_gni_table(url)

# rename cols before converting to polars
gni_df = gni_df.rename(columns={'Country': 'country'})
gni_df = pl.from_pandas(gni_df)

gni_df


In [ ]:
# normalize the colums for better merging
gni_df = gni_df.with_columns(normalize_column("country"))
base_df = base_df.with_columns(normalize_column("country"))


# left merge
main_df = base_df.join(gni_df.select(["gni_per_capita", "country_clean"]), on='country_clean', how='left')

# classify income groups using World Bank thresholds
main_df = main_df.with_columns(classify_income_group("gni_per_capita"))

main_df

In [ ]:
# carbon intensity: CO2 (millions of tonnes) per $1M GDP
main_df = main_df.with_columns(
    pl.when(pl.col("gdp") != 0)
      .then(pl.col("co2") * 1_000_000 / pl.col("gdp"))
      .otherwise(None)
      .alias("co2_per_gdp"))

# keep working from the enriched dataframe from here on
# this way later cells still have income groups and carbon intensity
base_df = main_df.clone()

main_df


In [ ]:

# fetch electricity production data from Our World in Data
df_electricity = pd.read_csv(
    "https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true",
    storage_options={'User-Agent': 'Our World In Data data fetch/1.0'}
)
df_electricity = pl.from_pandas(df_electricity)

# rename to concise column names
rename_map = {
    'code': 'iso_code',
    "other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked": "other_renewables",
    'bioenergy_generation__twh_chart_electricity_prod_source_stacked': 'bioenergy',
    'solar_generation__twh_chart_electricity_prod_source_stacked': 'solar',
    'wind_generation__twh_chart_electricity_prod_source_stacked': 'wind',
    'hydro_generation__twh_chart_electricity_prod_source_stacked': 'hydro',
    'nuclear_generation__twh_chart_electricity_prod_source_stacked': 'nuclear',
    'oil_generation__twh_chart_electricity_prod_source_stacked': 'oil',
    'gas_generation__twh_chart_electricity_prod_source_stacked': 'gas',
    'coal_generation__twh_chart_electricity_prod_source_stacked': 'coal',
}
df_electricity = df_electricity.rename(rename_map)

# keep only individual countries (drop aggregates without ISO codes)
df_electricity = df_electricity.filter(pl.col("iso_code").is_not_null())

df_electricity

In [ ]:
green_cols = ['other_renewables', 'bioenergy', 'solar', 'wind', 'hydro', 'nuclear']
non_green_cols = ['coal', 'oil', 'gas']

df_electricity = compute_energy_mix_shares(df_electricity, green_cols, non_green_cols)

df_electricity

In [ ]:
# start from the enriched dataframe, not the pre-income-group version
base_df = main_df.clone()

# columns coming from df_electricity (excluding merge keys and entity)
elec_cols = [c for c in df_electricity.columns if c not in ['entity', 'iso_code', 'year']]

# merge electricity data onto base_df
base_df = base_df.join(
    df_electricity.select(['iso_code', 'year'] + elec_cols),
    on=['iso_code', 'year'],
    how='inner'
)

base_df


# Electricity Mix and Carbon Intensity

Even today, coal remains one of the most widely used sources of electricity globally. Due to its large CO2 footprint, many countries struggle with pollution and carbon emissions.

Especially former Soviet republics and countries in the Central Asian region capitalise on their vast coal reserves and use them as their primary source of electricity. That, combined with generally smaller populations, typically leads to higher CO2 per GDP. This relationship between electricity mix and carbon intensity is explored below.

In [ ]:
# only choose valid values
required_cols = ['co2_per_gdp', 'non_green_share']
missing_efficiency_cols = [col for col in required_cols if col not in base_df.columns]
if missing_efficiency_cols:
    raise KeyError(f"Missing columns: {missing_efficiency_cols}. Re-run the carbon-intensity and electricity-merge cells above.")

valid_efficiency = base_df.filter(pl.col("co2_per_gdp") > 0)

# calculate avg 
efficiency_rank = (
    valid_efficiency
    .group_by("iso_code")
    .agg(pl.col("co2_per_gdp").mean().alias("mean_co2_per_gdp")))

# take the most and least efficient
top5_efficient = (
    efficiency_rank
    .sort("mean_co2_per_gdp")
    .head(5)
    .get_column("iso_code")
    .to_list())

top5_inefficient = (
    efficiency_rank
    .sort("mean_co2_per_gdp", descending=True)
    .head(5)
    .get_column("iso_code")
    .to_list())

countries_to_plot = top5_efficient + top5_inefficient

# prepare plot data
plot_df = (
    base_df
    .filter(pl.col("iso_code").is_in(countries_to_plot))
    .with_columns(
        pl.when(pl.col("iso_code").is_in(top5_efficient))
        .then(pl.lit("Most Efficient"))
        .otherwise(pl.lit("Least Efficient"))
        .alias("efficiency")))

# choose order
efficient_countries = (
    plot_df
    .filter(pl.col("efficiency") == "Most Efficient")
    .select("country")
    .unique(maintain_order=True)
    .get_column("country")
    .to_list())

inefficient_countries = (
    plot_df
    .filter(pl.col("efficiency") == "Least Efficient")
    .select("country")
    .unique(maintain_order=True)
    .get_column("country")
    .to_list())

col_order = efficient_countries + inefficient_countries

# convert to pandas for seaborn
plot_df_pd = plot_df.to_pandas()

g = sns.relplot(
    data=plot_df_pd,
    x="year",
    y="non_green_share",
    hue="efficiency",
    style="efficiency",
    col="country",
    col_wrap=5,
    col_order=col_order,
    kind="line",
    height=3,
    aspect=1.2,
    palette={"Most Efficient": "seagreen", "Least Efficient": "firebrick"},
    facet_kws={"xlim": (2000, 2024), "ylim": (0, 1.05)},)

g.set_titles("{col_name}")
g.set_axis_labels("Year", "Non-Green Electricity Share")
g.fig.suptitle(
    "Fossil Fuel Dependency: Most vs Least Carbon-Efficient Economies",
    y=1.02,
    fontsize=14,)

plt.show()


### Electricity Mix: Key Observations

Looking at the most and least efficient countries in terms of CO2 per GDP, a clear trend emerges: the more efficient a country is, the lower its share of non-green energy.

However, several anomalies stand out:

- **Bermuda** has a ~100% non-green electricity share but is deemed carbon-efficient. This is because its exceptionally high GDP (driven by financial services) means that its modest electricity emissions barely register per dollar of output.
- **South Sudan** appears efficient for the opposite reason - most of the population lacks grid electricity entirely and relies on burning wood or waste, neither of which is captured in this dataset.
- **Ukraine**, despite not having the highest share of non-green electricity, struggles with carbon efficiency due to ageing Soviet-era industrial infrastructure and, more recently, the economic impact of the ongoing war. A declining population(and economy) also pushes CO2 per GDP upward.

# Energy-Usage per capita

### Integrating External XML Data

To demonstrate XML data processing, an additional dataset was retrieved
from the World Bank API containing energy consumption per capita.

The XML structure was parsed and converted into a pandas DataFrame.
Relevant fields were extracted and merged with the main dataset.

Energy consumption is conceptually related to carbon emissions and
therefore provides an additional explanatory variable for later
machine learning models.

In [ ]:
# fetch xml data from worldbank.org(energy use per capita)
url = 'https://api.worldbank.org/v2/country/all/indicator/EG.USE.PCAP.KG.OE?format=xml&per_page=20000'

# pd.read_xml would have been possible too
# but parsing the xml ourselves makes the structure more explicit
energy_xml = fetch_world_bank_xml_records(url)

# only keep the fields we actually need for the merge later
energy_xml = energy_xml[[
    "countryiso3code",
    "date",
    "value"
]].rename(columns={
    "countryiso3code": "iso_code",
    "date": "year",
    "value": "energy_use_pc"}
    )

energy_xml = pl.from_pandas(energy_xml)

# years come in as text from the xml response
energy_xml = energy_xml.with_columns(pl.col("year")
                                     .cast(pl.Int64))

energy_xml


In [ ]:
# merge
main_df = main_df.join(
    energy_xml,
    on=["iso_code","year"],
    how="left")

main_df

In [ ]:
# mean the countries, so there are no duplicates from multiple years
mean_df = (
    main_df.group_by(["iso_code", "income_group"])
      .agg(pl.col("energy_use_pc").mean(),
           pl.col("co2_per_capita").mean(),
           pl.col("gdp_per_capita").mean()))


fig, ax = plt.subplots(figsize=(12, 6))

# plot a scatterplot to display energy use per capita and co2 per capita
sns.scatterplot(
    data=mean_df,
    x="energy_use_pc",
    y="co2_per_capita",
    hue="income_group",
    alpha=0.3,
    ax = ax)

# use a regression plot to show trends
sns.regplot(
    data=mean_df,
    x="energy_use_pc",
    y="co2_per_capita",
    scatter=False,
    color="red",
    ax = ax
)


plt.xlabel("Energy use per capita (kg oil equivalent)")
plt.ylabel("CO2 emissions per capita")
plt.title("Relationship between energy consumption and CO2 emissions")

plt.tight_layout()
plt.show()


# Correlation between energy usage and CO₂ emissions per capita

The country-level aggregation reveals a strong positive relationship between energy consumption per capita and CO₂ emissions per capita. This suggests that differences in energy demand across countries play a significant role in explaining variations in emissions.

High-income countries generally show higher energy consumption levels, while low-income countries cluster in the lower-left region of the plot. This pattern reflects the strong link between economic development, energy demand, and emissions.

To avoid distortions caused by extreme single-year observations, the dataset was aggregated at the country level by calculating mean values across all available years. This allows the visualisation to focus on structural differences between countries rather than short-term fluctuations.
